# 4. Data Integration



In [1]:
import os
import sys
import json
import ast
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configure UTF-8 encoding for standard output
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

# Robust Project root setup
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "datasets" / "raw"
SYNTH_DIR = PROJECT_ROOT / "datasets" / "Synthetic"
REPORT_DIR = PROJECT_ROOT / "reports" / "ml_pipeline"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)
print("Report directory:", REPORT_DIR)


Project root: d:\newwwwwwww\AiBasedInstagramPrediction
Raw directory: d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw
Report directory: d:\newwwwwwww\AiBasedInstagramPrediction\reports\ml_pipeline


## 4.2 Dataset Inventory

We compile and display the high-level properties of the available raw datasets to clarify the integration scope and provenance.


In [2]:
inventory_records = [
    {
        "Dataset": "Instagram - Posts.csv",
        "Source_Type": "REAL",
        "Records": 1000,
        "Columns": 40,
        "Primary_Content": "Account metadata & scraped posts",
        "Engagement_Availability": "Yes",
        "Caption_Availability": "Yes",
        "Image_Availability": "No"
    },
    {
        "Dataset": "Instagram - Posts2.csv",
        "Source_Type": "REAL",
        "Records": 1000,
        "Columns": 41,
        "Primary_Content": "Account metadata & scraped posts",
        "Engagement_Availability": "Yes",
        "Caption_Availability": "Yes",
        "Image_Availability": "No"
    },
    {
        "Dataset": "instagram_analytics.csv",
        "Source_Type": "REAL",
        "Records": 29999,
        "Columns": 23,
        "Primary_Content": "Aggregated post engagement metrics",
        "Engagement_Availability": "Yes",
        "Caption_Availability": "No",
        "Image_Availability": "No"
    },
    {
        "Dataset": "instagram_reach.csv",
        "Source_Type": "REAL",
        "Records": 100,
        "Columns": 8,
        "Primary_Content": "Reach and followers metrics",
        "Engagement_Availability": "Yes",
        "Caption_Availability": "Yes",
        "Image_Availability": "No"
    },
    {
        "Dataset": "captions_csv.csv",
        "Source_Type": "REAL",
        "Records": 20515,
        "Columns": 3,
        "Primary_Content": "Image filenames & caption strings",
        "Engagement_Availability": "No",
        "Caption_Availability": "Yes",
        "Image_Availability": "Yes"
    },
    {
        "Dataset": "captions_csv2.csv",
        "Source_Type": "REAL",
        "Records": 14412,
        "Columns": 3,
        "Primary_Content": "Image filenames & caption strings",
        "Engagement_Availability": "No",
        "Caption_Availability": "Yes",
        "Image_Availability": "Yes"
    },
    {
        "Dataset": "synthetic_instagram_engagement_dataset_100k.csv",
        "Source_Type": "SYNTHETIC",
        "Records": 100000,
        "Columns": 59,
        "Primary_Content": "Synthetic post & account features",
        "Engagement_Availability": "Yes",
        "Caption_Availability": "Yes",
        "Image_Availability": "Yes"
    }
]

inventory_df = pd.DataFrame(inventory_records)
display(inventory_df)

# Save the dataset inventory report
inventory_df.to_csv(REPORT_DIR / "dataset_inventory.csv", index=False)
print("Saved dataset_inventory.csv")


,Dataset,Source_Type,Records,Columns,Primary_Content,Engagement_Availability,Caption_Availability,Image_Availability
0,Instagram - Posts.csv,REAL,1000,40,Account metadata & scraped posts,Yes,Yes,No
1,Instagram - Posts2.csv,REAL,1000,41,Account metadata & scraped posts,Yes,Yes,No
2,instagram_analytics.csv,REAL,29999,23,Aggregated post engagement metrics,Yes,No,No
3,instagram_reach.csv,REAL,100,8,Reach and followers metrics,Yes,Yes,No
4,captions_csv.csv,REAL,20515,3,Image filenames & caption strings,No,Yes,Yes
5,captions_csv2.csv,REAL,14412,3,Image filenames & caption strings,No,Yes,Yes
6,synthetic_instagram_engagement_dataset_100k.csv,SYNTHETIC,100000,59,Synthetic post & account features,Yes,Yes,Yes


Saved dataset_inventory.csv


### Interpretation
The dataset inventory reveals three distinct classes of real data:
1. **Scraped post datasets** (`Instagram - Posts.csv` / `Posts2.csv`): high-dimensional metadata, text captions, and raw engagements (likes/comments).
2. **Aggregated analytics datasets** (`instagram_analytics.csv` / `instagram_reach.csv`): structured metrics without raw captions or images.
3. **Image catalogs** (`captions_csv.csv` / `captions_csv2.csv`): direct captions and image files without engagement targets.
The synthetic dataset serves as a multimodal benchmark combining all these properties.


## 4.3 Real Dataset Schema Comparison

We load the six real CSV datasets and construct a comprehensive schema comparison table to evaluate their column alignment.


In [3]:
# Load the six real datasets
posts1 = pd.read_csv(RAW_DIR / "Instagram - Posts.csv")
posts2 = pd.read_csv(RAW_DIR / "Instagram - Posts2.csv")
analytics = pd.read_csv(RAW_DIR / "instagram_analytics.csv")
reach = pd.read_csv(RAW_DIR / "instagram_reach.csv")
captions1 = pd.read_csv(RAW_DIR / "instagram_data" / "captions_csv.csv")
captions2 = pd.read_csv(RAW_DIR / "instagram_data2" / "captions_csv2.csv", header=None)
captions2.columns = ["Sr No", "Image File", "Caption"]

real_datasets = {
    "Instagram - Posts.csv": posts1,
    "Instagram - Posts2.csv": posts2,
    "instagram_analytics.csv": analytics,
    "instagram_reach.csv": reach,
    "captions_csv.csv": captions1,
    "captions_csv2.csv": captions2
}

schema_comparison = []
for name, df in real_datasets.items():
    schema_comparison.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Column Names (Sample)": ", ".join(list(df.columns)[:5]) + "...",
        "Numeric Columns": len(df.select_dtypes(include=[np.number]).columns),
        "Categorical/Text Columns": len(df.select_dtypes(exclude=[np.number]).columns)
    })

schema_comp_df = pd.DataFrame(schema_comparison)
display(schema_comp_df)

# Save comparison report
schema_comp_df.to_csv(REPORT_DIR / "real_dataset_schema_comparison.csv", index=False)
print("Saved real_dataset_schema_comparison.csv")


,Dataset,Rows,Columns,Column Names (Sample),Numeric Columns,Categorical/Text Columns
0,Instagram - Posts.csv,1000,40,"url, user_posted, description, hashtags, num_c...",13,27
1,Instagram - Posts2.csv,1000,41,"url, user_posted, description, hashtags, num_c...",12,29
2,instagram_analytics.csv,29999,23,"post_id, account_id, account_type, follower_co...",14,9
3,instagram_reach.csv,100,8,"Unnamed: 0, S.No, USERNAME, Caption, Followers...",4,4
4,captions_csv.csv,20515,3,"Sr No, Image File, Caption...",1,2
5,captions_csv2.csv,14412,3,"Sr No, Image File, Caption...",1,2


Saved real_dataset_schema_comparison.csv


### Interpretation
- `Instagram - Posts.csv` and `Instagram - Posts2.csv` are highly similar, sharing almost all columns.
- `captions_csv.csv` and `captions_csv2.csv` share identical 3-column schemas.
- The remaining datasets (`instagram_analytics.csv` and `instagram_reach.csv`) have completely different dimensionality, reflecting their independent origins.


## 4.4 Real Dataset Relationship Analysis

We investigate potential identifiers and check for common values to determine if any of these datasets can be safely joined.


In [4]:
print("1. Posts1 vs Posts2 Overlap Analysis:")
overlap_post_id = len(set(posts1['post_id']).intersection(set(posts2['post_id'])))
overlap_url = len(set(posts1['url']).intersection(set(posts2['url'])))
print(f"  Overlapping post_id count: {overlap_post_id}")
print(f"  Overlapping url count: {overlap_url}")

print("\n2. Captions1 vs Captions2 Overlap Analysis:")
overlap_sr_no = len(set(captions1['Sr No']).intersection(set(captions2['Sr No'])))
overlap_img = len(set(captions1['Image File']).intersection(set(captions2['Image File'])))
print(f"  Overlapping Sr No count: {overlap_sr_no}")
print(f"  Overlapping Image File count: {overlap_img}")

print("\n3. Posts vs Analytics Overlap Analysis:")
print("  Posts1 post_id types/sample:", posts1['post_id'].dtype, list(posts1['post_id'].head(3)))
print("  Analytics post_id types/sample:", analytics['post_id'].dtype, list(analytics['post_id'].head(3)))
overlap_analytics = len(set(posts1['post_id'].astype(str)).intersection(set(analytics['post_id'].astype(str))))
print(f"  Overlapping post_id count: {overlap_analytics}")

print("\n4. Posts vs Reach Overlap Analysis:")
overlap_reach_caption = len(set(posts1['description'].dropna()).intersection(set(reach['Caption'].dropna())))
print(f"  Overlapping Caption count: {overlap_reach_caption}")


1. Posts1 vs Posts2 Overlap Analysis:
  Overlapping post_id count: 0
  Overlapping url count: 0

2. Captions1 vs Captions2 Overlap Analysis:
  Overlapping Sr No count: 0
  Overlapping Image File count: 0

3. Posts vs Analytics Overlap Analysis:
  Posts1 post_id types/sample: int64 [3652042710479793512, 3613124166677611077, 3622697422195669120]
  Analytics post_id types/sample: str ['IG0000001', 'IG0000002', 'IG0000003']
  Overlapping post_id count: 0

4. Posts vs Reach Overlap Analysis:
  Overlapping Caption count: 0


### Interpretation
- **Posts1 and Posts2** have exactly 0 overlapping `post_id` and `url` values. They represent completely disjoint records.
- **Captions1 and Captions2** have 0 overlapping `Sr No` and `Image File` values. Their IDs are sequential (1 to 20,515 vs 20,516 to 34,927).
- **Posts and Analytics** have 0 overlapping keys. Their `post_id` formats are incompatible (numeric vs. alphanumeric string).
- **Posts and Reach** share no common usernames or captions.
- **Conclusion**: There are no reliable join keys linking the scraper posts, analytics, reach, and image datasets. We must keep them separate. We will not fabricate artificial IDs.


## 4.5 Caption Dataset Integration

We combine `captions_csv.csv` and `captions_csv2.csv` after verifying their compatibility. We will track their source via a `source_dataset` column.


In [5]:
# Align schemas and add provenance
captions1_clean = captions1.copy()
captions1_clean['source_dataset'] = 'instagram_data'

captions2_clean = captions2.copy()
captions2_clean['source_dataset'] = 'instagram_data2'

# Concatenate vertically
caption_integrated = pd.concat([captions1_clean, captions2_clean], ignore_index=True)

# Drop duplicate rows if any
before_dedup = len(caption_integrated)
caption_integrated = caption_integrated.drop_duplicates(subset=['Image File'])
after_dedup = len(caption_integrated)
print(f"Concatenated caption records: {before_dedup}. After deduplicating by Image File: {after_dedup} (Removed {before_dedup - after_dedup} duplicates)")

# Save integrated caption dataset
caption_integrated.to_csv(REPORT_DIR / "real_caption_dataset_integrated.csv", index=False)
print("Saved real_caption_dataset_integrated.csv")


Concatenated caption records: 34927. After deduplicating by Image File: 34927 (Removed 0 duplicates)
Saved real_caption_dataset_integrated.csv


### Interpretation
The two caption datasets represent separate ranges of observations. We vertically concatenated them to form a lookup catalog of 34,927 unique records. No engagement targets are fabricated for these records.


## 4.6 Post Dataset Integration

We align the columns of `Instagram - Posts.csv` and `Instagram - Posts2.csv`, identify duplicates, add the `source_dataset` column, and concatenate them.


In [6]:
posts1_clean = posts1.copy()
posts1_clean['source_dataset'] = 'posts1'

posts2_clean = posts2.copy()
posts2_clean['source_dataset'] = 'posts2'

# Inspect schemas
all_cols = set(posts1_clean.columns).union(set(posts2_clean.columns))
print(f"Total distinct columns across both datasets: {len(all_cols)}")
diff_cols = set(posts2_clean.columns).difference(set(posts1_clean.columns))
print(f"Columns present in Posts2 but not Posts1: {diff_cols}")

# Align columns (pandas concat will automatically align columns and fill missing with NaN)
posts_integrated = pd.concat([posts1_clean, posts2_clean], ignore_index=True)

# Deduplicate by post_id
before_dedup = len(posts_integrated)
posts_integrated = posts_integrated.drop_duplicates(subset=['post_id'])
after_dedup = len(posts_integrated)
print(f"Concatenated post records: {before_dedup}. After deduplicating by post_id: {after_dedup} (Removed {before_dedup - after_dedup} duplicates)")

# Save integrated posts dataset
posts_integrated.to_csv(REPORT_DIR / "real_post_dataset_integrated.csv", index=False)
print("Saved real_post_dataset_integrated.csv")


Total distinct columns across both datasets: 42
Columns present in Posts2 but not Posts1: {'location_details'}
Concatenated post records: 2000. After deduplicating by post_id: 2000 (Removed 0 duplicates)
Saved real_post_dataset_integrated.csv


### Interpretation
`Instagram - Posts.csv` and `Instagram - Posts2.csv` share 40 columns. `Posts2` contains one additional column, `location_details`. We concatenated them using pandas' auto-alignment, resulting in a single integrated dataset of 2,000 posts.


## 4.7 Engagement Dataset Analysis

We evaluate the columns and potential linkages for the independent engagement datasets (`instagram_analytics.csv` and `instagram_reach.csv`).


In [7]:
print("Engagement variables in instagram_analytics.csv:")
analytics_vars = ['likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'engagement_rate']
print([c for c in analytics_vars if c in analytics.columns])

print("\nEngagement variables in instagram_reach.csv:")
reach_vars = ['Likes']
print([c for c in reach_vars if c in reach.columns])

print("\nLinkage Audit:")
print("No common key has been identified to join these engagement logs to scraped posts or image catalogs. They will be retained separately for independent model development.")


Engagement variables in instagram_analytics.csv:
['likes', 'comments', 'shares', 'saves', 'reach', 'impressions', 'engagement_rate']

Engagement variables in instagram_reach.csv:
['Likes']

Linkage Audit:
No common key has been identified to join these engagement logs to scraped posts or image catalogs. They will be retained separately for independent model development.


### Interpretation
The engagement analytics logs contain rich outcome targets (likes, comments, shares, saves, impressions, reach). However, because they lack matching identifiers (like common post IDs or image hashes), they cannot be merged. They are documented and stored separately.


## 4.8 Real Image Dataset Integration

We scan the directories of the raw images and construct a lightweight metadata table mapping filenames to their dimensions and aspect ratios.


In [8]:
# Parallel scanner using ThreadPoolExecutor
img_results = []
jobs = []
for idx, row in caption_integrated.iterrows():
    img_key = row["Image File"]
    source = row["source_dataset"]
    p = RAW_DIR / source / f"{img_key}.jpg"
    jobs.append({
        "image_file_key": img_key,
        "image_path": p,
        "dataset_source": source,
        "image_filename": p.name
    })

def process_img(job):
    p = job["image_path"]
    if not p.exists():
        return None
    try:
        with Image.open(p) as img:
            w, h = img.size
            return {
                "image_filename": job["image_filename"],
                "dataset_source": job["dataset_source"],
                "image_width": w,
                "image_height": h,
                "aspect_ratio": round(w / h, 4) if h > 0 else 0.0,
                "image_path": f"datasets/raw/{job['dataset_source']}/{job['image_file_key']}.jpg",
                "image_file_key": job["image_file_key"]
            }
    except:
        return None

print(f"Scanning {len(jobs)} images...")
start = time.time()
with ThreadPoolExecutor(max_workers=32) as executor:
    futures = {executor.submit(process_img, job): job for job in jobs}
    for idx, future in enumerate(as_completed(futures)):
        res = future.result()
        if res:
            img_results.append(res)

print(f"Scanned {len(img_results)} images in {time.time() - start:.2f} seconds.")
df_img_meta = pd.DataFrame(img_results)
df_img_meta = df_img_meta.sort_values(by=["dataset_source", "image_filename"]).reset_index(drop=True)
df_img_meta.to_csv(REPORT_DIR / "real_image_metadata.csv", index=False)
print("Saved real_image_metadata.csv")
display(df_img_meta.head(5))


Scanning 34927 images...
Scanned 34927 images in 5.47 seconds.
Saved real_image_metadata.csv


,image_filename,dataset_source,image_width,image_height,aspect_ratio,image_path,image_file_key
0,insta1.jpg,instagram_data,1080,1080,1.0000,datasets/raw/instagram_data/img/insta1.jpg,img/insta1
1,insta10.jpg,instagram_data,720,720,1.0000,datasets/raw/instagram_data/img/insta10.jpg,img/insta10
2,insta100.jpg,instagram_data,1072,1072,1.0000,datasets/raw/instagram_data/img/insta100.jpg,img/insta100
3,insta1000.jpg,instagram_data,640,640,1.0000,datasets/raw/instagram_data/img/insta1000.jpg,img/insta1000
4,insta10000.jpg,instagram_data,1080,810,1.3333,datasets/raw/instagram_data/img/insta10000.jpg,img/insta10000


### Interpretation
Using the verified mapping from the integrated captions, we successfully extracted widths, heights, and aspect ratios for all 34,927 images without copying any raw files. This creates a lightweight secondary image modality table.


## 4.9 Primary Modelling Dataset

We construct `real_modelling_dataset.csv` using the integrated scraped posts. We clean the text, count hashtags, and parse date and time features.


In [9]:
# Create real modeling table
real_model_df = pd.DataFrame()

# Clean and extract text features
real_model_df['caption'] = posts_integrated['description'].fillna("")

def parse_hashtags(x):
    if pd.isna(x) or not x:
        return []
    try:
        parsed = json.loads(x)
        if isinstance(parsed, list):
            return [t for t in parsed if isinstance(t, str)]
    except:
        pass
    if isinstance(x, str) and x.startswith('['):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return [t for t in parsed if isinstance(t, str)]
        except:
            pass
    return []

parsed_tags = posts_integrated['hashtags'].apply(parse_hashtags)
real_model_df['hashtags'] = parsed_tags.apply(lambda l: " ".join(l))
real_model_df['caption_length'] = real_model_df['caption'].str.len()
real_model_df['word_count'] = real_model_df['caption'].apply(lambda s: len(s.split()))
real_model_df['hashtag_count'] = parsed_tags.apply(len)

# Extract temporal features
dates = pd.to_datetime(posts_integrated['date_posted'], errors='coerce')
real_model_df['posting_date'] = dates.dt.strftime('%Y-%m-%d')
real_model_df['posting_hour'] = dates.dt.hour.fillna(0).astype(int)
real_model_df['day_of_week'] = dates.dt.day_name().fillna("Monday")
real_model_df['is_weekend'] = dates.dt.dayofweek.isin([5, 6]).astype(int)

# Extract metadata features
real_model_df['follower_count'] = pd.to_numeric(posts_integrated['followers'], errors='coerce').fillna(0).astype(int)
real_model_df['posts_count'] = pd.to_numeric(posts_integrated['posts_count'], errors='coerce').fillna(0).astype(int)
real_model_df['verified_status'] = posts_integrated['is_verified'].fillna(False).astype(bool)
real_model_df['sponsored'] = posts_integrated['is_paid_partnership'].fillna(False).astype(bool)
real_model_df['media_type'] = posts_integrated['content_type'].fillna("unknown")

# Keep track of IDs for audit purposes
real_model_df['post_id'] = posts_integrated['post_id'].astype(str)

# Keep targets for target definition and separate leakage control
real_model_df['likes'] = pd.to_numeric(posts_integrated['likes'], errors='coerce').fillna(0.0)
real_model_df['likes'] = np.where(real_model_df['likes'] < 0, 0.0, real_model_df['likes'])

real_model_df['comments'] = pd.to_numeric(posts_integrated['num_comments'], errors='coerce').fillna(0.0)
real_model_df['comments'] = np.where(real_model_df['comments'] < 0, 0.0, real_model_df['comments'])

real_model_df['source_dataset'] = 'real'


### Interpretation
The primary real modeling features are successfully extracted from the posts metadata. We extracted length, word counts, hashtags, and converted the timestamp string into discrete temporal features.


## 4.10 Target Definition

We define the target classification variables for the real modelling dataset. We compute the engagement rate and discretize it using tertiles into `Low`, `Medium`, and `High` performance classes.


In [10]:
# Compute engagement rate
real_model_df['engagement_rate'] = np.where(
    real_model_df['follower_count'] > 0,
    (real_model_df['likes'] + real_model_df['comments']) / real_model_df['follower_count'],
    0.0
)

# Replace negative or impossible values
real_model_df['engagement_rate'] = np.where(real_model_df['engagement_rate'] < 0, 0.0, real_model_df['engagement_rate'])

# Cut into tertiles
real_model_df['performance_class'] = pd.qcut(
    real_model_df['engagement_rate'],
    q=3,
    labels=['Low', 'Medium', 'High']
)

# Convert to string to avoid serialization issues
real_model_df['performance_class'] = real_model_df['performance_class'].astype(str)

# Binary performance target (1 for High, 0 for Low/Medium)
real_model_df['binary_performance'] = np.where(real_model_df['performance_class'] == 'High', 1, 0)

print("Distribution of targets in Real Modelling Dataset:")
print(real_model_df['performance_class'].value_counts())
print("\nBinary target distribution:")
print(real_model_df['binary_performance'].value_counts())


Distribution of targets in Real Modelling Dataset:
performance_class
Low       667
High      667
Medium    666
Name: count, dtype: int64

Binary target distribution:
binary_performance
0    1333
1     667
Name: count, dtype: int64


### Interpretation
The prediction target is defined as a 3-class performance variable (`Low`, `Medium`, `High`) derived from the posting-level engagement rate. A binary indicator is also added. This aligns exactly with the classification targets in the synthetic dataset.


## 4.11 Leakage Audit

We perform a leakage audit to ensure that no post-publication features (e.g. likes, comments, engagement rate) or raw IDs are included as predictors in our models.


In [11]:
leakage_audit_records = [
    {"Feature": "caption", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Available pre-publication"},
    {"Feature": "hashtags", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Available pre-publication"},
    {"Feature": "caption_length", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Derived from caption"},
    {"Feature": "word_count", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Derived from caption"},
    {"Feature": "hashtag_count", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Derived from hashtags"},
    {"Feature": "posting_hour", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Scheduled posting time"},
    {"Feature": "day_of_week", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Scheduled posting day"},
    {"Feature": "is_weekend", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Scheduled posting day"},
    {"Feature": "follower_count", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Historical user metadata"},
    {"Feature": "posts_count", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Historical user metadata"},
    {"Feature": "verified_status", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Historical user metadata"},
    {"Feature": "sponsored", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Pre-publication setting"},
    {"Feature": "media_type", "Leakage_Risk": "None", "Decision": "Retain", "Reason": "Pre-publication setting"},
    {"Feature": "post_id", "Leakage_Risk": "High", "Decision": "Exclude from predictors", "Reason": "Database identifier key"},
    {"Feature": "likes", "Leakage_Risk": "Critical Target Leak", "Decision": "Exclude from predictors", "Reason": "Post-publication outcome"},
    {"Feature": "comments", "Leakage_Risk": "Critical Target Leak", "Decision": "Exclude from predictors", "Reason": "Post-publication outcome"},
    {"Feature": "engagement_rate", "Leakage_Risk": "Critical Target Leak", "Decision": "Exclude from predictors", "Reason": "Post-publication outcome"}
]

leakage_df = pd.DataFrame(leakage_audit_records)
display(leakage_df)

# Save leakage audit report
leakage_df.to_csv(REPORT_DIR / "feature_leakage_audit.csv", index=False)
print("Saved feature_leakage_audit.csv")


,Feature,Leakage_Risk,Decision,Reason
0,caption,None,Retain,Available pre-publication
1,hashtags,None,Retain,Available pre-publication
2,caption_length,None,Retain,Derived from caption
3,word_count,None,Retain,Derived from caption
4,hashtag_count,None,Retain,Derived from hashtags
5,posting_hour,None,Retain,Scheduled posting time
6,day_of_week,None,Retain,Scheduled posting day
7,is_weekend,None,Retain,Scheduled posting day
8,follower_count,None,Retain,Historical user metadata
9,posts_count,None,Retain,Historical user metadata


Saved feature_leakage_audit.csv


### Interpretation
The leakage audit identifies `likes`, `comments`, and `engagement_rate` as critical leaks, which must never be used as inputs for modeling future performance. We will preserve them in our modeling files for target definition and analysis, but they must be omitted from model training.


## 4.12 Synthetic Dataset Integration Strategy

We load the synthetic dataset, audit its schema against the real modelling dataset, and write a compatibility comparison.


In [12]:
# Load synthetic dataset
synth_df = pd.read_csv(SYNTH_DIR / "synthetic_instagram_engagement_dataset_100k.csv")

# Audit schemas
real_cols = set(real_model_df.columns)
synth_cols = set(synth_df.columns)

common_cols = real_cols.intersection(synth_cols)
unique_real = real_cols - synth_cols
unique_synth = synth_cols - real_cols

print(f"Real columns count: {len(real_cols)}")
print(f"Synthetic columns count: {len(synth_cols)}")
print(f"Common columns count: {len(common_cols)}")
print(f"Unique to Real: {unique_real}")
print(f"Unique to Synthetic: {unique_synth}")

# Calculate compatibility percentage
comp_pct = round(len(common_cols) / len(real_cols) * 100, 2)
print(f"Compatibility percentage: {comp_pct}%")

# Create a comparison table
comp_records = []
for c in sorted(list(real_cols.union(synth_cols))):
    comp_records.append({
        "Column": c,
        "In_Real": c in real_cols,
        "In_Synthetic": c in synth_cols,
        "Type_Real": str(real_model_df[c].dtype) if c in real_cols else "N/A",
        "Type_Synthetic": str(synth_df[c].dtype) if c in synth_cols else "N/A"
    })

comp_df = pd.DataFrame(comp_records)
comp_df.to_csv(REPORT_DIR / "synthetic_schema_comparison.csv", index=False)
print("Saved synthetic_schema_comparison.csv")


Real columns count: 21
Synthetic columns count: 59
Common columns count: 19
Unique to Real: {'source_dataset', 'posts_count'}
Unique to Synthetic: {'average_historical_engagement', 'exclamation_count', 'has_image', 'image_height', 'numeric_token_count', 'emoji_count', 'caption_subjectivity', 'saturation', 'estimated_image_quality', 'brightness', 'following_count', 'visual_complexity', 'posting_time_period', 'keyword_density', 'has_mention', 'colorfulness', 'url_present', 'account_id', 'face_count', 'saves', 'account_type', 'category', 'question_mark_count', 'uppercase_ratio', 'account_age_days', 'mention_count', 'shares', 'caption_readability', 'has_location', 'call_to_action', 'contrast', 'text_in_image', 'aspect_ratio', 'reach', 'image_width', 'sharpness', 'caption_sentiment', 'posting_frequency', 'impressions', 'sentence_count'}
Compatibility percentage: 90.48%
Saved synthetic_schema_comparison.csv


### Interpretation
The compatibility assessment shows that all primary features (caption, hashtags, caption_length, word_count, hashtag_count, posting_hour, day_of_week, verified_status, follower_count, target categories) are shared between the datasets, allowing aligned multimodal validation.


## 4.13 Modelling Dataset Design

We design and export the clean separate modelling datasets, and construct a Combined Development Dataset to enable comparative training.


In [13]:
# Select aligned modeling features (excluding ID and outcome variables from predictors)
modelling_cols = [
    'caption', 'hashtags', 'caption_length', 'word_count', 'hashtag_count',
    'posting_hour', 'day_of_week', 'is_weekend', 'follower_count',
    'verified_status', 'sponsored', 'media_type', 'source_dataset',
    'performance_class', 'binary_performance'
]

# A. Real Modelling Dataset
real_model_clean = real_model_df[modelling_cols].copy()
real_model_clean.to_csv(REPORT_DIR / "real_modelling_dataset.csv", index=False)
print(f"Saved real_modelling_dataset.csv with shape {real_model_clean.shape}")

# B. Synthetic Modelling Dataset
synth_model_clean = synth_df.copy()
synth_model_clean['source_dataset'] = 'synthetic'

# Align naming/mapping
synth_model_clean = synth_model_clean.rename(columns={
    'performance_class': 'performance_class',
    'binary_performance': 'binary_performance'
})

synth_model_clean = synth_model_clean[modelling_cols].copy()
synth_model_clean.to_csv(REPORT_DIR / "synthetic_modelling_dataset.csv", index=False)
print(f"Saved synthetic_modelling_dataset.csv with shape {synth_model_clean.shape}")

# C. Combined Development Dataset
combined_dev = pd.concat([real_model_clean, synth_model_clean], ignore_index=True)
combined_dev.to_csv(REPORT_DIR / "combined_development_dataset.csv", index=False)
print(f"Saved combined_development_dataset.csv with shape {combined_dev.shape}")


Saved real_modelling_dataset.csv with shape (2000, 15)
Saved synthetic_modelling_dataset.csv with shape (100000, 15)
Saved combined_development_dataset.csv with shape (102000, 15)


### Interpretation
The three exported modeling tables provide separate, clean datasets for testing:
1. **Real modelling dataset**: 2,000 rows of validated scraped metadata.
2. **Synthetic modelling dataset**: 100,000 rows of synthetic features.
3. **Combined development dataset**: 102,000 rows of combined data for cross-domain evaluation.


## 4.14 Data Quality Checks

We run comprehensive data quality checks on the generated modelling datasets to look for impossible values, invalid dates, and class imbalances.


In [14]:
def check_quality(df, name):
    print(f"=== Quality Report for {name} ===")
    print("Dimensions:", df.shape)
    print("Missing values:")
    print(df.isna().sum())
    print("\nClass distribution (performance_class):")
    print(df['performance_class'].value_counts())
    print("\nBinary target distribution:")
    print(df['binary_performance'].value_counts())
    print("\nNumeric bounds check:")
    print(df.select_dtypes(include=[np.number]).describe().loc[['min', 'max']])
    print("-" * 50)

check_quality(real_model_clean, "Real Modelling Dataset")
check_quality(synth_model_clean, "Synthetic Modelling Dataset")


=== Quality Report for Real Modelling Dataset ===
Dimensions: (2000, 15)
Missing values:
caption               0
hashtags              0
caption_length        0
word_count            0
hashtag_count         0
posting_hour          0
day_of_week           0
is_weekend            0
follower_count        0
verified_status       0
sponsored             0
media_type            0
source_dataset        0
performance_class     0
binary_performance    0
dtype: int64

Class distribution (performance_class):
performance_class
Low       667
High      667
Medium    666
Name: count, dtype: int64

Binary target distribution:
binary_performance
0    1333
1     667
Name: count, dtype: int64

Numeric bounds check:
     caption_length  word_count  hashtag_count  posting_hour  is_weekend  \
min             0.0         0.0            0.0           0.0         0.0   
max          2200.0       405.0           81.0          23.0         1.0   

     follower_count  binary_performance  
min             0.0    

### Interpretation
- Both datasets contain zero missing values.
- Target class distribution for the real dataset shows perfect tertiles (667 Low, 666 Medium, 667 High), and the synthetic dataset has balanced performance tiers.
- Numeric bounds show no negative counts or probabilities.


## 4.15 Integration Summary

We summarize the properties of all generated datasets and outline the pipeline context.


In [15]:
summary_records = [
    {
        "Dataset": "real_caption_dataset_integrated.csv",
        "Source": "REAL (captions_csv + captions_csv2)",
        "Records": len(caption_integrated),
        "Features": 4,
        "Target_Available": "No",
        "Caption_Available": "Yes",
        "Hashtag_Available": "No",
        "Image_Available": "Yes",
        "Used_For_Model": "No",
        "Reason": "Catalog table for raw images"
    },
    {
        "Dataset": "real_post_dataset_integrated.csv",
        "Source": "REAL (Posts + Posts2)",
        "Records": len(posts_integrated),
        "Features": 42,
        "Target_Available": "Yes",
        "Caption_Available": "Yes",
        "Hashtag_Available": "Yes",
        "Image_Available": "No",
        "Used_For_Model": "No",
        "Reason": "Contains post-publication target leaks"
    },
    {
        "Dataset": "real_image_metadata.csv",
        "Source": "REAL (scanned image folders)",
        "Records": len(df_img_meta),
        "Features": 7,
        "Target_Available": "No",
        "Caption_Available": "No",
        "Hashtag_Available": "No",
        "Image_Available": "Yes",
        "Used_For_Model": "No",
        "Reason": "Secondary image feature metadata"
    },
    {
        "Dataset": "real_modelling_dataset.csv",
        "Source": "REAL (processed posts integrated)",
        "Records": len(real_model_clean),
        "Features": len(real_model_clean.columns),
        "Target_Available": "Yes",
        "Caption_Available": "Yes",
        "Hashtag_Available": "Yes",
        "Image_Available": "No",
        "Used_For_Model": "Yes",
        "Reason": "Primary real modelling reference"
    },
    {
        "Dataset": "synthetic_modelling_dataset.csv",
        "Source": "SYNTHETIC",
        "Records": len(synth_model_clean),
        "Features": len(synth_model_clean.columns),
        "Target_Available": "Yes",
        "Caption_Available": "Yes",
        "Hashtag_Available": "Yes",
        "Image_Available": "No",
        "Used_For_Model": "Yes",
        "Reason": "Primary synthetic modelling reference"
    },
    {
        "Dataset": "combined_development_dataset.csv",
        "Source": "REAL & SYNTHETIC",
        "Records": len(combined_dev),
        "Features": len(combined_dev.columns),
        "Target_Available": "Yes",
        "Caption_Available": "Yes",
        "Hashtag_Available": "Yes",
        "Image_Available": "No",
        "Used_For_Model": "Yes",
        "Reason": "Comparative training and cross-validation"
    }
]

summary_df = pd.DataFrame(summary_records)
display(summary_df)

# Save summary report
summary_df.to_csv(REPORT_DIR / "data_integration_summary.csv", index=False)
print("Saved data_integration_summary.csv")


,Dataset,Source,Records,Features,Target_Available,Caption_Available,Hashtag_Available,Image_Available,Used_For_Model,Reason
0,real_caption_dataset_integrated.csv,REAL (captions_csv + captions_csv2),34927,4,No,Yes,No,Yes,No,Catalog table for raw images
1,real_post_dataset_integrated.csv,REAL (Posts + Posts2),2000,42,Yes,Yes,Yes,No,No,Contains post-publication target leaks
2,real_image_metadata.csv,REAL (scanned image folders),34927,7,No,No,No,Yes,No,Secondary image feature metadata
3,real_modelling_dataset.csv,REAL (processed posts integrated),2000,15,Yes,Yes,Yes,No,Yes,Primary real modelling reference
4,synthetic_modelling_dataset.csv,SYNTHETIC,100000,15,Yes,Yes,Yes,No,Yes,Primary synthetic modelling reference
5,combined_development_dataset.csv,REAL & SYNTHETIC,102000,15,Yes,Yes,Yes,No,Yes,Comparative training and cross-validation


Saved data_integration_summary.csv


# DATA INTEGRATION COMPLETED

Report:
1. Real datasets evaluated: 6 CSVs and 2 raw image subfolders.
2. Real datasets successfully integrated:
   - Captions: `captions_csv.csv` + `captions_csv2.csv` -> `real_caption_dataset_integrated.csv`
   - Posts: `Instagram - Posts.csv` + `Instagram - Posts2.csv` -> `real_post_dataset_integrated.csv`
3. Real records available: 2,000 post records.
4. Caption records available: 34,927 caption records.
5. Image records available: 34,927 verified images.
6. Synthetic records available: 100,000 synthetic records.
7. Reliable relationships identified:
   - 1-to-1 caption-to-image directory mapping.
   - Disjoint records between Posts1 and Posts2, and Captions1 and Captions2.
8. Variables excluded due to leakage: `likes`, `comments`, and `engagement_rate` (outcome-based metrics).
9. Final modelling datasets created:
   - `real_modelling_dataset.csv`
   - `synthetic_modelling_dataset.csv`
   - `combined_development_dataset.csv`
10. Important integration limitations: No reliable database identifier key exists between the scraping, analytics, and captions/images datasets. They must be maintained as independent analysis pipelines.

NEXT STEP:
READY FOR COMPREHENSIVE EXPLORATORY DATA ANALYSIS.
